# 01 · Quality control and preprocessing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/en/01_quality_control_preprocessing.ipynb)

**From Cells to Clusters: An Introduction to Single-Cell RNA-seq Analysis in Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Peru · November 9, 2026  
Instructor: Dr. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Time in the tutorial:** 9:40–10:15 (35 min)

**By the end of this notebook you will be able to:**
- Compute and plot per-cell QC metrics (UMIs, genes, % mitochondrial, % ribosomal)
- Choose filtering thresholds: fixed cut-offs vs. data-driven (MAD) cut-offs
- Detect and remove doublets with Scrublet
- Normalize, log-transform and select highly variable genes

> 🟢 **Core** sections are covered live. 🔎 **Extra** sections are for self-study after the tutorial.

## 0. Setup 🟢
Run the cell below first (▶ or `Shift + Enter`). On Google Colab it installs the packages (~1 min);
on your own computer it assumes you already ran `pip install -r requirements.txt`.

**Optional:** set `USE_DRIVE = True` to save your results in Google Drive, so the next notebooks
can reuse them. If you leave it `False`, each notebook rebuilds what it needs automatically.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10"

USE_DRIVE = False  # True = keep results in Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Tutorial helper functions (download + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # running from a local clone
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

## 1. Load the Cell Ranger matrix 🟢
`tut.load_pbmc3k_raw()` does what we did step by step in notebook 00 (download + `sc.read_10x_mtx`).

In [ ]:
adata = tut.load_pbmc3k_raw(WORKDIR)
adata

## 2. Why quality control? 🟢
Not every barcode is a healthy single cell:

| Problem | Signature | Why it happens |
|---|---|---|
| **Dying / broken cell** | few genes, high **% mitochondrial** reads | cytoplasmic mRNA leaks out, mitochondrial transcripts stay |
| **Empty droplet** with ambient RNA | very few UMIs and genes | Cell Ranger kept it as a cell |
| **Doublet** (two cells in one droplet) | unusually many UMIs/genes, mixed markers | ~0.8 % per 1,000 cells loaded |
| Red blood cell contamination | high **hemoglobin** (`HBA1`, `HBB`) | common in blood/tissue samples |

We flag them with a few numbers per cell.

In [ ]:
# Flag gene groups (human symbols; for mouse use "mt-", ("Rps", "Rpl"), "^Hb[^(p)]")
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains(r"^HB[^(P)]")
print(adata.var[["mt", "ribo", "hb"]].sum())

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)
adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"]].describe().round(2)

### Look at the distributions 🟢

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="pct_counts_mt")

**How to read it:** healthy cells lie on the diagonal (more UMIs → more genes). Cells in the **bottom-left** with
**high % MT** (yellow) are low quality. Cells far to the **top-right** may be doublets.

## 3. Choosing thresholds 🟢
**Option A: fixed cut-offs.** Simple and common for well-known samples. For PBMC 3k the classic choice
(Seurat and Scanpy tutorials) is: ≥ 200 genes, < 2,500 genes, < 5 % mitochondrial.

**Option B: data-driven cut-offs (MAD).** Flag cells more than *k* **median absolute deviations** away from the
median. It adapts to each sample, which is useful when you have many samples or tissues with different
baselines (e.g. neurons or cardiomyocytes have naturally higher % MT).

In [ ]:
def is_outlier(adata, metric, nmads):
    x = adata.obs[metric]
    mad = np.median(np.abs(x - np.median(x)))
    return (x < np.median(x) - nmads * mad) | (x > np.median(x) + nmads * mad)

adata.obs["outlier_mad"] = (
    is_outlier(adata, "log1p_total_counts", 5)
    | is_outlier(adata, "log1p_n_genes_by_counts", 5)
    | is_outlier(adata, "pct_counts_mt", 3)
)
adata.obs["outlier_fixed"] = (
    (adata.obs.n_genes_by_counts < 200) | (adata.obs.n_genes_by_counts >= 2500) | (adata.obs.pct_counts_mt >= 5)
)
pd.crosstab(adata.obs["outlier_fixed"], adata.obs["outlier_mad"],
            rownames=["fixed cut-offs"], colnames=["MAD cut-offs"])

Both strategies mostly agree. We continue with the **fixed** cut-offs so your numbers match the classic
tutorials. Always report which thresholds you used in your methods section!

In [ ]:
n0 = adata.n_obs
sc.pp.filter_cells(adata, min_genes=200)   # cells with < 200 genes
sc.pp.filter_genes(adata, min_cells=3)     # genes seen in < 3 cells
adata = adata[(adata.obs.n_genes_by_counts < 2500) & (adata.obs.pct_counts_mt < 5)].copy()
print(f"cells: {n0} -> {adata.n_obs}   genes: {adata.n_vars}")

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)

## 4. Doublets with Scrublet 🟢
Scrublet simulates artificial doublets by adding random pairs of cells, then scores every real cell by how similar
it is to the simulated doublets. It works **per sample** (use `batch_key="sample"` when you have several).

In [ ]:
sc.pp.scrublet(adata, random_state=0)
print(adata.obs["predicted_doublet"].value_counts())
sc.pl.scrublet_score_distribution(adata)

In [ ]:
adata = adata[~adata.obs["predicted_doublet"]].copy()
print("cells after doublet removal:", adata.n_obs)

> 🔎 **Extra: ambient RNA.** Free-floating mRNA from lysed cells ends up in every droplet. Tools such as
> **SoupX** (R), **CellBender** (Python, GPU) or **scAR** estimate this "soup" from the `raw_feature_bc_matrix`
> and subtract it. It matters most for tissues with very abundant transcripts (e.g. epithelia, hemoglobin in blood).

## 5. Normalization 🟢
Cells were sequenced to different depths, so raw counts are not comparable. The standard approach:
1. **Keep the raw counts** in a layer (we need them later for pseudobulk differential expression).
2. **Scale** each cell to the same total (10,000 counts = "counts per 10k").
3. **Log-transform**, `log(x + 1)`, to reduce the effect of very highly expressed genes.

Alternatives: scran pooling, analytic Pearson residuals (`sc.experimental.pp.normalize_pearson_residuals`),
or `SCTransform` in Seurat.

In [ ]:
adata.layers["counts"] = adata.X.copy()   # raw counts, never modified
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata  # snapshot of the log-normalized values of ALL genes

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(np.asarray(adata.layers["counts"].sum(1)).ravel(), bins=50, color="#264653")
axes[0].set(title="raw counts per cell", xlabel="total")
axes[1].hist(np.asarray(adata.X.sum(1)).ravel(), bins=50, color="#2a9d8f")
axes[1].set(title="after normalize + log1p", xlabel="total")
plt.tight_layout()

## 6. Highly variable genes (HVGs) 🟢
Most genes are either not expressed or constant across cells: they add noise but no information.
We keep the ~2,000 genes whose variance is higher than expected for their mean expression.
They are only **flagged** (`adata.var["highly_variable"]`); all genes stay in the object.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pl.highly_variable_genes(adata)
print(adata.var.query("highly_variable").sort_values("dispersions_norm", ascending=False).head(15).index.tolist())

## 7. Save the checkpoint 🟢

In [ ]:
os.makedirs(f"{WORKDIR}/results", exist_ok=True)
adata.write_h5ad(f"{WORKDIR}/results/pbmc3k_qc.h5ad")
adata

### 💡 Your turn
Reload the raw data (`tut.load_pbmc3k_raw(WORKDIR)`) and count how many cells would pass if you allowed
up to **10 %** mitochondrial reads instead of 5 %. Is that a big difference for PBMCs?

In [ ]:
# Your code here

<details><summary><b>Show solution</b></summary>

```python
raw = tut.load_pbmc3k_raw(WORKDIR)
raw.var["mt"] = raw.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(raw, qc_vars=["mt"], inplace=True)
for thr in (5, 10):
    keep = (raw.obs.n_genes_by_counts >= 200) & (raw.obs.n_genes_by_counts < 2500) & (raw.obs.pct_counts_mt < thr)
    print(thr, "% MT ->", keep.sum(), "cells")
```
</details>

---
## ✅ Done!
Your data is clean and normalized. Next: **02 · Dimensionality reduction and clustering**.